# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 300

In [3]:
df = pd.read_csv(f"../../../data/top30groups/LongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [6]:
geodata = ['longitude', 'latitude']
combined_geo = df.copy()
combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
combined_geo = combined_geo.drop(columns=geodata)

In [7]:
import ast

def to_tuple_if_needed(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val  # already a tuple

combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [8]:
torch.cuda.empty_cache()


In [9]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds = []
y_trues = []
logs = []

# Default config (from previous best)
default_args = {
    'partition': f"gtd{partition}",
    #'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    #'n_tree': 80,
    'tree_depth': 10,
    #'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': False
}

import os

embed_dim_grid = [8, 16, 32]
n_tree_grid = [40, 80, 120]
feature_rate_grid = [0.1, 0.3, 0.5]

for col in continuous_cols:
    print(f"\nRunning grid search for {col} prediction...")

    data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
        combined_geo, label_index, continuous_col=col)

    best_overall_acc = -1.0
    best_config = {}

    for embed_dim in embed_dim_grid:
        for n_tree in n_tree_grid:
            for tree_feature_rate in feature_rate_grid:
                print(f"\nTraining with embed_dim={embed_dim}, n_tree={n_tree}, tree_feature_rate={tree_feature_rate}")

                args = default_args.copy()
                args['embed_dim'] = embed_dim
                args['n_tree'] = n_tree
                args['tree_feature_rate'] = tree_feature_rate

                best_acc, best_epoch, best_precision, best_recall, best_f1, y_pred_decoded, y_true_decoded, \
                best_precision_micro, best_recall_micro, best_f1_micro, best_precision_macro, best_recall_macro, best_f1_macro, \
                roc_auc_weighted, roc_auc_micro, roc_auc_macro, epoch_logs = train_joint(
                    data, data.edge_index, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
                    args, row_to_node_index, index_to_label, verbose=True)

                y_preds.append(y_pred_decoded)
                y_trues.append(y_true_decoded)
                logs.append(epoch_logs)

                os.makedirs(f"Results{partition}", exist_ok=True)

                results_path = f"Results{partition}/Results_{col}_embed{embed_dim}_nTree{n_tree}_fRate{tree_feature_rate}"
                with open(results_path, "w") as f:
                    f.write(f"Best acc: {best_acc:.4f} at epoch {best_epoch} for {col} prediction\n")
                    f.write(f"Weighted Precision: {best_precision:.4f}, Recall: {best_recall:.4f}, F1: {best_f1:.4f}\n")
                    f.write(f"Macro Precision: {best_precision_macro:.4f}, Recall: {best_recall_macro:.4f}, F1: {best_f1_macro:.4f}\n")
                    f.write(f"Micro Precision: {best_precision_micro:.4f}, Recall: {best_recall_micro:.4f}, F1: {best_f1_micro:.4f}\n")
                    f.write(f"AUROC Weighted: {roc_auc_weighted:.4f}, Micro: {roc_auc_micro:.4f}, Macro: {roc_auc_macro:.4f}\n")

                log_path = f"Results{partition}/epoch_logs_{col}_embed{embed_dim}_nTree{n_tree}_fRate{tree_feature_rate}"
                with open(log_path, "w") as f:
                    f.write('\n'.join(f"{x:.4f}" for x in epoch_logs))

                if best_acc > best_overall_acc:
                    best_overall_acc = best_acc
                    best_config = {
                        'embed_dim': embed_dim,
                        'n_tree': n_tree,
                        'tree_feature_rate': tree_feature_rate,
                        'best_acc': best_acc,
                        'epoch': best_epoch,
                        'macro_f1': best_f1_macro,
                        'micro_f1': best_f1_micro,
                        'weighted_f1': best_f1
                    }

    # Save best config
    summary_path = f"Results{partition}/BestConfig_{col}.txt"
    with open(summary_path, "w") as f:
        f.write(f"Best configuration for {col} prediction:\n")
        for k, v in best_config.items():
            f.write(f"{k}: {v}\n")





Running grid search for weaptype1 prediction...

Training with embed_dim=8, n_tree=40, tree_feature_rate=0.1
Epoch 01 | GCN MSE Loss: 1.2631 | NRF Loss: 3.4012 | JOINT Loss: 4.6644 | NRF Acc: 0.1671
Epoch 51 | GCN MSE Loss: 1.1337 | NRF Loss: 3.2537 | JOINT Loss: 4.3874 | NRF Acc: 0.3936
Epoch 101 | GCN MSE Loss: 1.1045 | NRF Loss: 3.0308 | JOINT Loss: 4.1353 | NRF Acc: 0.5183
Epoch 151 | GCN MSE Loss: 1.0932 | NRF Loss: 2.8387 | JOINT Loss: 3.9319 | NRF Acc: 0.6526
Epoch 201 | GCN MSE Loss: 1.0875 | NRF Loss: 2.6707 | JOINT Loss: 3.7583 | NRF Acc: 0.7423
Epoch 251 | GCN MSE Loss: 1.0835 | NRF Loss: 2.5201 | JOINT Loss: 3.6036 | NRF Acc: 0.7857
Epoch 301 | GCN MSE Loss: 1.0800 | NRF Loss: 2.3824 | JOINT Loss: 3.4623 | NRF Acc: 0.8104
Epoch 351 | GCN MSE Loss: 1.0765 | NRF Loss: 2.2560 | JOINT Loss: 3.3325 | NRF Acc: 0.8179
Epoch 401 | GCN MSE Loss: 1.0738 | NRF Loss: 2.1411 | JOINT Loss: 3.2149 | NRF Acc: 0.8325
Epoch 451 | GCN MSE Loss: 1.0724 | NRF Loss: 2.0345 | JOINT Loss: 3.1069 

KeyboardInterrupt: 

In [ ]:
best_args = default_args.copy()
best_args['embed_dim'] = best_config['embed_dim']
best_args['n_tree'] = best_config['n_tree']
best_args['tree_feature_rate'] = best_config['tree_feature_rate']
best_args['final_evaluation'] = True  # now do full test evaluation
best_args['epochs'] = 3000

# You may want to reload the data again here
data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
    combined_geo, label_index, continuous_col='weaptype1')

test_acc, best_epoch, best_precision, best_recall, best_f1, y_pred_decoded, y_true_decoded, best_precision_micro, best_recall_micro, best_f1_micro, best_precision_macro, best_recall_macro, best_f1_macro, roc_auc_weighted, roc_auc_micro, roc_auc_macro, epoch_logs = train_joint(
    data, data.edge_index, y_gcn, y_nrf, non_geo_features,
    train_mask, val_mask, test_mask,
    best_args, row_to_node_index, index_to_label, verbose=True
)


Epoch 01 | GCN MSE Loss: 1.1989 | NRF Loss: 3.4012 | JOINT Loss: 4.6001 | NRF Acc: 0.3998
Epoch 51 | GCN MSE Loss: 1.0275 | NRF Loss: 3.0553 | JOINT Loss: 4.0828 | NRF Acc: 0.6034
Epoch 101 | GCN MSE Loss: 1.0190 | NRF Loss: 2.8668 | JOINT Loss: 3.8858 | NRF Acc: 0.7818
Epoch 151 | GCN MSE Loss: 1.0141 | NRF Loss: 2.7173 | JOINT Loss: 3.7314 | NRF Acc: 0.8154
Epoch 201 | GCN MSE Loss: 1.0101 | NRF Loss: 2.5814 | JOINT Loss: 3.5916 | NRF Acc: 0.8456
Epoch 251 | GCN MSE Loss: 1.0072 | NRF Loss: 2.4568 | JOINT Loss: 3.4640 | NRF Acc: 0.8549
Epoch 301 | GCN MSE Loss: 1.0052 | NRF Loss: 2.3396 | JOINT Loss: 3.3448 | NRF Acc: 0.8845
Epoch 351 | GCN MSE Loss: 1.0034 | NRF Loss: 2.2310 | JOINT Loss: 3.2344 | NRF Acc: 0.8926
Epoch 401 | GCN MSE Loss: 1.0018 | NRF Loss: 2.1300 | JOINT Loss: 3.1318 | NRF Acc: 0.8951
Epoch 451 | GCN MSE Loss: 1.0003 | NRF Loss: 2.0358 | JOINT Loss: 3.0360 | NRF Acc: 0.8951
Epoch 501 | GCN MSE Loss: 0.9987 | NRF Loss: 1.9473 | JOINT Loss: 2.9460 | NRF Acc: 0.8982
E

In [ ]:
test_acc

0.9114004373550415

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

'\nBest acc: 0.9205 at epoch 750 for weaptype1 prediction\nWeighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191\nMacro Precision: 0.9176, Recall: 0.9107, F1: 0.9105\nMicro Precision: 0.9205, Recall: 0.9205, F1: 0.9205\nAUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963\n\n'

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       1.00      0.94      0.97        98
        African National Congress (South Africa)       0.99      1.00      1.00       126
                                Al-Qaida in Iraq       0.95      0.74      0.83       180
        Al-Qaida in the Arabian Peninsula (AQAP)       0.93      0.87      0.90       127
                                      Al-Shabaab       1.00      1.00      1.00       116
             Basque Fatherland and Freedom (ETA)       1.00      1.00      1.00       126
                                      Boko Haram       0.98      0.96      0.97        92
  Communist Party of India - Maoist (CPI-Maoist)       0.98      0.92      0.95        65
       Corsican National Liberation Front (FLNC)       1.00      0.99      1.00       145
                       Donetsk People's Republic       0.99      0.99      0.99       115
Farabundo

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 200 to Results200/cm_200_weaptype1.png
